# Chapter 7 — Adapter & Facade Patterns
## *Being Adaptive*

Two structural patterns that deal with interfaces.

---

## Adapter Pattern

**Intent:** Convert the interface of a class into another interface that clients expect.  
Adapter lets classes work together that couldn't otherwise because of incompatible interfaces.

### OO Design Principle
- **Principle of Least Knowledge** (introduced later, but relevant here): only talk to close friends.

### Book context
A `Duck` interface vs a `Turkey` interface — make a Turkey behave like a Duck.

In [ ]:
from abc import ABC, abstractmethod

# ── Existing target interface ─────────────────────────────────────────────────
class Duck(ABC):
    @abstractmethod
    def quack(self): ...
    @abstractmethod
    def fly(self): ...

class MallardDuck(Duck):
    def quack(self): print("Quack!")
    def fly(self):   print("I'm flying a long way!")


# ── Adaptee: incompatible interface ──────────────────────────────────────────
class Turkey(ABC):
    @abstractmethod
    def gobble(self): ...
    @abstractmethod
    def fly(self): ...

class WildTurkey(Turkey):
    def gobble(self): print("Gobble gobble!")
    def fly(self):    print("I'm flying a short distance.")


# ── Object Adapter ────────────────────────────────────────────────────────────
class TurkeyAdapter(Duck):
    """Makes a Turkey look like a Duck."""
    def __init__(self, turkey: Turkey):
        self._turkey = turkey

    def quack(self):  # translate quack → gobble
        self._turkey.gobble()

    def fly(self):    # turkeys fly short — compensate
        for _ in range(5):
            self._turkey.fly()


def duck_test(duck: Duck):
    duck.quack()
    duck.fly()

print("=== Real Duck ===")
duck_test(MallardDuck())

print("\n=== Turkey via Adapter ===")
turkey_adapter = TurkeyAdapter(WildTurkey())
duck_test(turkey_adapter)

## Real-world Adapter example — legacy payment gateway

In [ ]:
# Modern interface our app expects
class PaymentProcessor(ABC):
    @abstractmethod
    def charge(self, amount: float, currency: str) -> bool: ...


# Legacy SDK we can't modify
class LegacyPaymentSDK:
    def make_payment(self, amount_cents: int, currency_code: str) -> str:
        print(f"Legacy SDK: charging {amount_cents}¢ in {currency_code}")
        return "OK"


# Adapter
class LegacyPaymentAdapter(PaymentProcessor):
    def __init__(self, sdk: LegacyPaymentSDK):
        self._sdk = sdk

    def charge(self, amount: float, currency: str) -> bool:
        result = self._sdk.make_payment(int(amount * 100), currency.upper())
        return result == "OK"


adapter = LegacyPaymentAdapter(LegacyPaymentSDK())
success = adapter.charge(29.99, "usd")
print(f"Payment success: {success}")

---

## Facade Pattern

**Intent:** Provide a simplified interface to a complex subsystem.

### OO Design Principle
- **Principle of Least Knowledge (Law of Demeter):** Only talk to your immediate friends.  
  `a.b().c().d()` violates it — you become coupled to the whole chain.

### Book context
Home theater: `Amplifier`, `DVDPlayer`, `Projector`, `Screen`, `Lights` — a `HomeTheaterFacade` hides all the startup/shutdown complexity.

In [ ]:
# ── Subsystem classes ─────────────────────────────────────────────────────────
class Amplifier:
    def on(self):         print("Amp on")
    def set_volume(self, v): print(f"Amp volume → {v}")
    def off(self):        print("Amp off")

class DVDPlayer:
    def on(self):           print("DVD player on")
    def play(self, movie):  print(f"DVD playing '{movie}'")
    def stop(self):         print("DVD stopped")
    def off(self):          print("DVD player off")

class Projector:
    def on(self):        print("Projector on")
    def wide_screen(self): print("Projector → widescreen mode")
    def off(self):       print("Projector off")

class TheaterLights:
    def dim(self, v):  print(f"Lights dimmed to {v}%")
    def on(self):      print("Lights on")

class Screen:
    def down(self):  print("Screen down")
    def up(self):    print("Screen up")


# ── Facade ────────────────────────────────────────────────────────────────────
class HomeTheaterFacade:
    def __init__(self, amp: Amplifier, dvd: DVDPlayer,
                 proj: Projector, lights: TheaterLights, screen: Screen):
        self._amp    = amp
        self._dvd    = dvd
        self._proj   = proj
        self._lights = lights
        self._screen = screen

    def watch_movie(self, movie: str):
        print(f"\nGet ready to watch {movie}!")
        self._lights.dim(10)
        self._screen.down()
        self._proj.on()
        self._proj.wide_screen()
        self._amp.on()
        self._amp.set_volume(5)
        self._dvd.on()
        self._dvd.play(movie)

    def end_movie(self):
        print("\nShutting down theater...")
        self._dvd.stop()
        self._dvd.off()
        self._amp.off()
        self._proj.off()
        self._screen.up()
        self._lights.on()


theater = HomeTheaterFacade(
    Amplifier(), DVDPlayer(), Projector(), TheaterLights(), Screen()
)
theater.watch_movie("Raiders of the Lost Ark")
theater.end_movie()

## Interview cheat-sheet

| | Adapter | Facade |
|---|---|---|
| **Intent** | Convert an interface | Simplify a subsystem |
| **How many classes** | Wraps one class | Coordinates many classes |
| **Client knows subsystem?** | No | Can still access it directly |
| **Analogy** | Power plug adapter | Hotel concierge |

**Real-world Python examples:**
- Adapter: `io.TextIOWrapper` wraps binary streams with a text interface.
- Facade: `requests.get()` hides `socket`, `http.client`, `urllib` complexity.

**Pattern family:** Structural (both)